#  VideoMAE

In [ ]:
!pip install -q transformers accelerate decord

import os
import cv2
import torch
import random
import numpy as np
import pandas as pd

from tqdm import tqdm

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import (
    VideoMAEImageProcessor,
    VideoMAEForVideoClassification
)

# =========================================================
# DEVICE
# =========================================================

device = "cuda" if torch.cuda.is_available() else "cpu"

print(device)

# =========================================================
# PATHS
# =========================================================

VIDEO_DIR = "/kaggle/input/datasets/rimmajeed/final-224-cropped-videos/final_224x224_cropped_videos/final_224x224_cropped_videos"

CSV_PATH = "/kaggle/input/datasets/rimmajeed/final-224-cropped-videos/final_224x224_labels.csv"

# =========================================================
# LOAD CSV
# =========================================================

df = pd.read_csv(CSV_PATH)

df["Labels"] = (
    df["Labels"]
    .str.strip()
    .str.lower()
)

# =========================================================
# CLASSES
# =========================================================

class_names = [
    "cheating_copying",
    "cheating_gesture",
    "cheating_mobile",
    "cheating_notes",
    "normal"
]

label2id = {
    c:i for i,c in enumerate(class_names)
}

id2label = {
    i:c for c,i in label2id.items()
}

df = df[df["Labels"].isin(class_names)]

df = df.reset_index(drop=True)

# =========================================================
# SPLIT
# =========================================================

train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["Labels"],
    random_state=42
)

print("Train:", len(train_df))
print("Val:", len(val_df))

# =========================================================
# PROCESSOR
# =========================================================

processor = VideoMAEImageProcessor.from_pretrained(
    "MCG-NJU/videomae-base-finetuned-kinetics"
)

# =========================================================
# VIDEO LOADER
# =========================================================

def load_video(path, num_frames=16):

    cap = cv2.VideoCapture(path)

    frames = []

    while True:

        ret, frame = cap.read()

        if not ret:
            break

        frame = cv2.cvtColor(
            frame,
            cv2.COLOR_BGR2RGB
        )

        frame = cv2.resize(
            frame,
            (224,224)
        )

        frames.append(frame)

    cap.release()

    if len(frames) == 0:
        return None

    # temporal sampling

    indices = np.linspace(
        0,
        len(frames)-1,
        num_frames
    ).astype(int)

    sampled = [
        frames[i]
        for i in indices
    ]

    return sampled

# =========================================================
# DATASET
# =========================================================

class ExamDataset(Dataset):

    def __init__(self, df):

        self.df = df

    def __len__(self):

        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        path = os.path.join(
            VIDEO_DIR,
            row["Videos"]
        )

        frames = load_video(path)

        if frames is None:

            frames = [
                np.zeros((224,224,3),dtype=np.uint8)
                for _ in range(16)
            ]

        encoding = processor(
            frames,
            return_tensors="pt"
        )

        pixel_values = encoding[
            "pixel_values"
        ].squeeze(0)

        label = label2id[
            row["Labels"]
        ]

        return pixel_values, label

# =========================================================
# DATALOADER
# =========================================================

train_dataset = ExamDataset(train_df)

val_dataset = ExamDataset(val_df)

train_loader = DataLoader(
    train_dataset,
    batch_size=2,
    shuffle=True,
    num_workers=2
)

val_loader = DataLoader(
    val_dataset,
    batch_size=2,
    shuffle=False,
    num_workers=2
)

# =========================================================
# MODEL
# =========================================================

model = VideoMAEForVideoClassification.from_pretrained(
    "MCG-NJU/videomae-base-finetuned-kinetics",
    num_labels=5,
    ignore_mismatched_sizes=True,
    label2id=label2id,
    id2label=id2label
)

model.to(device)

# =========================================================
# FREEZE EARLY LAYERS
# =========================================================

for name, param in model.named_parameters():

    param.requires_grad = False

    if (
        "encoder.layer.10" in name
        or
        "encoder.layer.11" in name
        or
        "classifier" in name
    ):

        param.requires_grad = True

# =========================================================
# FOCAL LOSS
# =========================================================

class FocalLoss(nn.Module):

    def __init__(self, gamma=2):

        super().__init__()

        self.gamma = gamma

    def forward(self, inputs, targets):

        ce = nn.functional.cross_entropy(
            inputs,
            targets,
            reduction='none'
        )

        pt = torch.exp(-ce)

        loss = (
            (1 - pt) ** self.gamma
        ) * ce

        return loss.mean()

criterion = FocalLoss()

# =========================================================
# OPTIMIZER
# =========================================================

optimizer = torch.optim.AdamW(
    filter(
        lambda p: p.requires_grad,
        model.parameters()
    ),
    lr=2e-5,
    weight_decay=1e-4
)

# =========================================================
# TRAINING WITH:
# 1. Train Loss
# 2. Train Accuracy
# 3. Validation Loss
# 4. Validation Accuracy
# 5. F1 Score
# 6. Early Stopping Patience
# 7. Best Model Saving
# =========================================================

from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score

# =========================================================
# SETTINGS
# =========================================================

EPOCHS = 25

PATIENCE = 3

best_acc = 0

counter = 0

# =========================================================
# STORE HISTORY
# =========================================================

train_losses = []
val_losses = []

train_accs = []
val_accs = []

train_f1s = []
val_f1s = []

# =========================================================
# TRAIN LOOP
# =========================================================

for epoch in range(EPOCHS):

    # =====================================================
    # TRAIN
    # =====================================================

    model.train()

    train_preds = []
    train_labels = []

    train_loss = 0

    for pixel_values, labels in tqdm(
        train_loader,
        desc=f"Train Epoch {epoch+1}"
    ):

        pixel_values = pixel_values.to(device)

        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(
            pixel_values=pixel_values
        )

        logits = outputs.logits

        loss = criterion(
            logits,
            labels
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            1.0
        )

        optimizer.step()

        train_loss += loss.item()

        preds = torch.argmax(
            logits,
            dim=1
        )

        train_preds.extend(
            preds.cpu().numpy()
        )

        train_labels.extend(
            labels.cpu().numpy()
        )

    # =====================================================
    # TRAIN METRICS
    # =====================================================

    avg_train_loss = (
        train_loss / len(train_loader)
    )

    train_acc = accuracy_score(
        train_labels,
        train_preds
    )

    train_f1 = f1_score(
        train_labels,
        train_preds,
        average="weighted"
    )

    # =====================================================
    # VALIDATION
    # =====================================================

    model.eval()

    val_preds = []
    val_labels = []

    val_loss = 0

    with torch.no_grad():

        for pixel_values, labels in tqdm(
            val_loader,
            desc=f"Val Epoch {epoch+1}"
        ):

            pixel_values = pixel_values.to(device)

            labels = labels.to(device)

            outputs = model(
                pixel_values=pixel_values
            )

            logits = outputs.logits

            loss = criterion(
                logits,
                labels
            )

            val_loss += loss.item()

            preds = torch.argmax(
                logits,
                dim=1
            )

            val_preds.extend(
                preds.cpu().numpy()
            )

            val_labels.extend(
                labels.cpu().numpy()
            )

    # =====================================================
    # VALIDATION METRICS
    # =====================================================

    avg_val_loss = (
        val_loss / len(val_loader)
    )

    val_acc = accuracy_score(
        val_labels,
        val_preds
    )

    val_f1 = f1_score(
        val_labels,
        val_preds,
        average="weighted"
    )

    # =====================================================
    # STORE HISTORY
    # =====================================================

    train_losses.append(avg_train_loss)
    val_losses.append(avg_val_loss)

    train_accs.append(train_acc)
    val_accs.append(val_acc)

    train_f1s.append(train_f1)
    val_f1s.append(val_f1)

    # =====================================================
    # PRINT ALL IN ONE LINE
    # =====================================================

    print(

        f"Epoch [{epoch+1}/{EPOCHS}] | "

        f"Train Loss: {avg_train_loss:.4f} | "

        f"Train Acc: {train_acc:.4f} | "

        f"Train F1: {train_f1:.4f} | "

        f"Val Loss: {avg_val_loss:.4f} | "

        f"Val Acc: {val_acc:.4f} | "

        f"Val F1: {val_f1:.4f}"

    )
EPOCHS = 25
    # =====================================================
    # SAVE BEST MODEL
    # =====================================================

    if val_acc > best_acc:

        best_acc = val_acc

        counter = 0

        torch.save(
            model.state_dict(),
            "/kaggle/working/best_videomae.pth"
        )

        print("Best Model Saved")

    else:

        counter += 1

        print(
            f"No Improvement | Patience: {counter}/{PATIENCE}"
        )

    # =====================================================
    # EARLY STOPPING
    # =====================================================

    if counter >= PATIENCE:

        print("\nEarly Stopping Triggered")

        break

# =========================================================
# FINAL RESULT
# =========================================================

print("\nTraining Finished")

print(f"\nBest Validation Accuracy: {best_acc:.4f}")

IndentationError: unexpected indent (665013937.py, line 525)

In [ ]:
# =========================================================
# FINAL EVALUATION CELL
# =========================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import torch

# =========================================================
# LOAD BEST MODEL
# =========================================================

model.load_state_dict(
    torch.load(
        "/kaggle/working/best_videomae.pth"
    )
)

model.eval()

# =========================================================
# EVALUATION FUNCTION
# =========================================================

def evaluate_model(
    model,
    loader,
    criterion,
    device,
    class_names,
    set_name="Validation"
):

    model.eval()

    all_preds = []
    all_labels = []

    total_loss = 0

    with torch.no_grad():

        for pixel_values, labels in loader:

            pixel_values = pixel_values.to(device)

            labels = labels.to(device)

            # =========================
            # FORWARD
            # =========================

            outputs = model(
                pixel_values=pixel_values
            )

            logits = outputs.logits

            # =========================
            # LOSS
            # =========================

            loss = criterion(
                logits,
                labels
            )

            total_loss += loss.item()

            # =========================
            # PREDICTIONS
            # =========================

            preds = torch.argmax(
                logits,
                dim=1
            )

            all_preds.extend(
                preds.cpu().numpy()
            )

            all_labels.extend(
                labels.cpu().numpy()
            )

    # =====================================================
    # METRICS
    # =====================================================

    avg_loss = total_loss / len(loader)

    acc = accuracy_score(
        all_labels,
        all_preds
    )

    precision = precision_score(
        all_labels,
        all_preds,
        average='weighted',
        zero_division=0
    )

    recall = recall_score(
        all_labels,
        all_preds,
        average='weighted',
        zero_division=0
    )

    f1 = f1_score(
        all_labels,
        all_preds,
        average='weighted',
        zero_division=0
    )

    # =====================================================
    # PRINT RESULTS
    # =====================================================

    print(f"\n{'='*60}")
    print(f"{set_name.upper()} RESULTS")
    print(f"{'='*60}")

    print(f"Loss      : {avg_loss:.4f}")
    print(f"Accuracy  : {acc:.4f}")
    print(f"Precision : {precision:.4f}")
    print(f"Recall    : {recall:.4f}")
    print(f"F1 Score  : {f1:.4f}")

    # =====================================================
    # CLASSIFICATION REPORT
    # =====================================================

    print("\nClassification Report:\n")

    report = classification_report(
        all_labels,
        all_preds,
        target_names=class_names,
        digits=4
    )

    print(report)

    # =====================================================
    # SAVE REPORT
    # =====================================================

    with open(
        f"/kaggle/working/{set_name.lower()}_classification_report.txt",
        "w"
    ) as f:

        f.write(report)

    # =====================================================
    # CONFUSION MATRIX
    # =====================================================

    cm = confusion_matrix(
        all_labels,
        all_preds
    )

    plt.figure(figsize=(8,6))

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=class_names
    )

    disp.plot(
        cmap="Blues",
        xticks_rotation=45
    )

    plt.title(f"{set_name} Confusion Matrix")

    plt.savefig(
        f"/kaggle/working/{set_name.lower()}_confusion_matrix.png",
        dpi=300,
        bbox_inches='tight'
    )

    plt.show()

    # =====================================================
    # RETURN METRICS
    # =====================================================

    return {

        "loss": avg_loss,
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

# =========================================================
# EVALUATE TRAIN
# =========================================================

train_results = evaluate_model(
    model=model,
    loader=train_loader,
    criterion=criterion,
    device=device,
    class_names=class_names,
    set_name="Train"
)

# =========================================================
# EVALUATE VALIDATION
# =========================================================

val_results = evaluate_model(
    model=model,
    loader=val_loader,
    criterion=criterion,
    device=device,
    class_names=class_names,
    set_name="Validation"
)

# =========================================================
# FINAL SUMMARY
# =========================================================

print("\n" + "="*70)
print("FINAL SUMMARY")
print("="*70)

summary_df = pd.DataFrame({

    "Dataset": ["Train", "Validation"],

    "Loss": [
        train_results["loss"],
        val_results["loss"]
    ],

    "Accuracy": [
        train_results["accuracy"],
        val_results["accuracy"]
    ],

    "Precision": [
        train_results["precision"],
        val_results["precision"]
    ],

    "Recall": [
        train_results["recall"],
        val_results["recall"]
    ],

    "F1 Score": [
        train_results["f1"],
        val_results["f1"]
    ]
})

print(summary_df)

# =========================================================
# SAVE SUMMARY CSV
# =========================================================

summary_df.to_csv(
    "/kaggle/working/VideoMAE_metrics_summary.csv",
    index=False
)

print("\nAll evaluation files saved successfully.")
print("/kaggle/working/")

In [ ]:
# =========================================================
# TRAINING CURVES
# =========================================================

epochs_range = range(1, len(train_losses)+1)

# ---------------- LOSS CURVE ----------------

plt.figure(figsize=(10,6))

plt.plot(
    epochs_range,
    train_losses,
    marker='o',
    label="Train Loss"
)

plt.plot(
    epochs_range,
    val_losses,
    marker='o',
    label="Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.grid(True)

loss_curve_path = "/kaggle/working/VideoMAE_loss_curve.png"

plt.savefig(
    loss_curve_path,
    dpi=300,
    bbox_inches='tight'
)

plt.show()

print("VideMAE Loss Curve Saved")

# ---------------- ACCURACY CURVE ----------------

plt.figure(figsize=(10,6))

plt.plot(
    epochs_range,
    train_accs,
    marker='o',
    label="Train Accuracy"
)

plt.plot(
    epochs_range,
    val_accs,
    marker='o',
    label="Validation Accuracy"
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training vs Validation Accuracy")
plt.legend()
plt.grid(True)

acc_curve_path = "/kaggle/working/VideoMAE_accuracy_curve.png"

plt.savefig(
    acc_curve_path,
    dpi=300,
    bbox_inches='tight'
)

plt.show()

print("Accuracy Curve Saved")

# =========================================================
# F1 SCORE CURVE
# =========================================================

if "train_f1s" in globals():

    plt.figure(figsize=(10,6))

    plt.plot(
        epochs_range,
        train_f1s,
        marker='o',
        label="Train F1"
    )

    plt.plot(
        epochs_range,
        val_f1s,
        marker='o',
        label="Validation F1"
    )

    plt.xlabel("Epoch")
    plt.ylabel("F1 Score")
    plt.title("Train vs Validation F1 Score")
    plt.legend()
    plt.grid(True)

    f1_curve_path = "/kaggle/working/VideoMAE f1_curve.png"

    plt.savefig(
        f1_curve_path,
        dpi=300,
        bbox_inches='tight'
    )

    plt.show()

    print("F1 Curve Saved")

# =========================================================
# CONFUSION MATRIX
# =========================================================

cm = confusion_matrix(
    val_results["labels"],
    val_results["preds"]
)

plt.figure(figsize=(8,8))

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=class_names
)

disp.plot(
    xticks_rotation=45,
    cmap="Blues"
)

plt.title("Validation Confusion Matrix")

cm_path = "/kaggle/working/VideoMAE confusion_matrix.png"

plt.savefig(
    cm_path,
    dpi=300,
    bbox_inches='tight'
)

plt.show()

print("Confusion Matrix Saved")

# =========================================================
# NORMALIZED CONFUSION MATRIX
# =========================================================

cm_norm = confusion_matrix(
    val_results["labels"],
    val_results["preds"],
    normalize='true'
)

plt.figure(figsize=(8,8))

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_norm,
    display_labels=class_names
)

disp.plot(
    xticks_rotation=45,
    cmap="Blues",
    values_format=".2f"
)

plt.title("Normalized Confusion Matrix")

norm_cm_path = "/kaggle/working/VideoMAE normalized_confusion_matrix.png"

plt.savefig(
    norm_cm_path,
    dpi=300,
    bbox_inches='tight'
)

plt.show()

print("Normalized Confusion Matrix Saved")

# =========================================================
# SAVE FINAL MODEL
# =========================================================

final_model_path = "/kaggle/working/final_videomae_model.pth"

torch.save(
    model.state_dict(),
    final_model_path
)

print("\nFinal Model Saved")

# =========================================================
# FINAL SUMMARY
# =========================================================

print("\n" + "="*60)
print("FINAL VALIDATION PERFORMANCE")
print("="*60)

print(f"Accuracy  : {val_results['accuracy']:.4f}")
print(f"Precision : {val_results['precision']:.4f}")
print(f"Recall    : {val_results['recall']:.4f}")
print(f"F1 Score  : {val_results['f1']:.4f}")

print("\nAll files saved in:")
print("/kaggle/working/")

## Testing On Whole class video

In [ ]:
# 1- Video Prediction
def predict_video(video_path):
    model.eval()

    clip = load_video(video_path)
    if clip is None:
        return "Invalid", 0

    clip = clip.unsqueeze(0).to(device)

    with torch.no_grad():
        out = model(clip)
        prob = torch.softmax(out, dim=1)[0].cpu().numpy()

    pred = np.argmax(prob)

    return idx_to_class[pred], prob[pred]

In [ ]:
# # Test on single video
# video = "/content/drive/MyDrive/ourDataset/dataset_f3/c11.mp4"

# pred, conf = predict_video(video)

# print("Prediction:", pred)
# print("Confidence:", round(float(conf),2))

In [ ]:
!pip install ultralytics opencv-python torch torchvision

In [ ]:
import cv2
import os
import torch
import numpy as np
import pandas as pd
from ultralytics import YOLO
from torchvision.models.video import r3d_18
from collections import deque

In [ ]:
# load model
device = "cuda" if torch.cuda.is_available() else "cpu"

# YOLO detector + tracker
MODEL_NAME = "yolov8x.pt"
yolo = YOLO(MODEL_NAME)

# Cheating classifier
# Use the UntrimmedNet architecture that was trained and saved
model = UntrimmedNet(num_classes=5)

model.load_state_dict(
    torch.load("/kaggle/working/I3D_UntrimmedNet_224.pth")
)

model = model.to(device)
model.eval()

# The class_names and idx_to_class are already defined earlier and are in scope.

In [ ]:
# step no 4: Frame --> Clip Function (critcal part)
IMG_SIZE = 224
CLIP_LEN = 64

mean = np.array([0.43216,0.394666,0.37645]).reshape(3,1,1,1)
std  = np.array([0.22803,0.22145,0.216989]).reshape(3,1,1,1)

def make_clip(frames):

    clip = []

    for f in frames:

        f = cv2.resize(f,(IMG_SIZE,IMG_SIZE))
        f = cv2.cvtColor(f,cv2.COLOR_BGR2RGB)

        f = np.array(f).astype(np.float32) / 255.0

        clip.append(f)

    clip = np.array(clip)

    clip = np.transpose(clip,(3,0,1,2))  # C,T,H,W

    clip = (clip - mean) / std

    clip = torch.tensor(clip,dtype=torch.float32)

    return clip.unsqueeze(0).to(device)


In [ ]:
# step 5: Predict clip
def predict_clip(clip):

    with torch.no_grad():
        out = model(clip)
        prob = torch.softmax(out,dim=1)[0]
        cls = torch.argmax(prob).item()

    return idx_to_class[cls], float(prob[cls])


In [ ]:
# Upload whole class video
from google.colab import files

# print("Upload WHOLE CLASS video:")
# uploaded = files.upload()

# video_name = list(uploaded.keys())[0]

video_name = "/content/drive/MyDrive/OurDataset/dataset_f4/DSC_0267.MOV"
# logs = analyze_class_video(video_name)


In [ ]:
# Step 6: Process whole class video (FINAL VERSION)

import pandas as pd
from collections import deque, defaultdict

def analyze_class_video(video_path, save_csv_path):

    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)

    student_buffers = {}
    tracker_to_student = {}

    prediction_buffer = defaultdict(list)

    SMOOTH_WINDOW = 5
    CONF_THRESHOLD = 0.6

    student_counter = 0
    frame_no = 0

    if not os.path.exists(save_csv_path):
        pd.DataFrame(columns=[
            "student_id",
            "start_time_s",
            "end_time_s",
            "cheating_type",
            "confidence"
        ]).to_csv(save_csv_path, index=False)

    while True:

        ret, frame = cap.read()
        if not ret:
            break

        h, w, _ = frame.shape

        results = yolo.track(frame, persist=True, classes=[0], conf=0.4, verbose=False)

        if results[0].boxes.id is not None:

            boxes = results[0].boxes.xyxy.cpu().numpy()
            ids = results[0].boxes.id.cpu().numpy().astype(int)

            for box, track_id in zip(boxes, ids):

                # ✅ Assign student ID
                if track_id not in tracker_to_student:
                    student_counter += 1
                    tracker_to_student[track_id] = student_counter

                sid = tracker_to_student[track_id]

                # ✅ IMPORTANT: Get box coords
                x1, y1, x2, y2 = map(int, box)

                # Clamp
                x1 = max(0, x1); y1 = max(0, y1)
                x2 = min(w, x2); y2 = min(h, y2)

                box_w = x2 - x1
                box_h = y2 - y1

                # ❌ Ignore small detections
                if box_w < 80 or box_h < 80:
                    continue

                # ❌ Ignore border-touching (partial persons)
                margin = 10
                if x1 <= margin or y1 <= margin or x2 >= (w - margin) or y2 >= (h - margin):
                    continue

                # ❌ Ignore bad aspect ratios
                aspect_ratio = box_h / (box_w + 1e-5)
                if aspect_ratio < 0.5 or aspect_ratio > 3.0:
                    continue

                # ✅ Add padding (VERY IMPORTANT)
                pad = 20
                x1 = max(0, x1 - pad)
                y1 = max(0, y1 - pad)
                x2 = min(w, x2 + pad)
                y2 = min(h, y2 + pad)

                crop = frame[y1:y2, x1:x2]

                if crop.size == 0:
                    continue

                # ✅ Buffer init
                if sid not in student_buffers:
                    student_buffers[sid] = deque(maxlen=CLIP_LEN)

                student_buffers[sid].append(crop)

                # ✅ Process every few frames (efficiency)
                if frame_no % 8 == 0 and len(student_buffers[sid]) == CLIP_LEN:
                    clip = make_clip(list(student_buffers[sid]))
                    label, conf = predict_clip(clip)

                    # ❌ Confidence filtering
                    if conf < CONF_THRESHOLD:
                        label = "normal"

                    # ✅ Smooth predictions
                    prediction_buffer[sid].append(label)

                    if len(prediction_buffer[sid]) > SMOOTH_WINDOW:
                        prediction_buffer[sid].pop(0)

                    final_label = max(
                        set(prediction_buffer[sid]),
                        key=prediction_buffer[sid].count
                    )

                    start_t = frame_no / fps
                    end_t = (frame_no + CLIP_LEN) / fps

                    row = {
                        "student_id": sid,
                        "start_time_s": round(start_t, 2),
                        "end_time_s": round(end_t, 2),
                        "cheating_type": final_label,
                        "confidence": round(conf, 2)
                    }

                    pd.DataFrame([row]).to_csv(
                        save_csv_path,
                        mode='a',
                        header=False,
                        index=False
                    )

        if frame_no % 200 == 0:
            print("Processed frame:", frame_no)

        frame_no += 1

    cap.release()
    print("Finished processing video")

In [ ]:
# Log of step 6: chunk file run
video_name = "/content/drive/MyDrive/OurDataset/dataset_f4/DSC_0267.MOV"

csv_path = "/content/drive/MyDrive/My_Models/Logs/i3d_DSC_0267_log.csv"

analyze_class_video(video_name,csv_path)

In [ ]:
# Step 8: Review Log from csv
import pandas as pd

csv_path = "/content/drive/MyDrive/My_Models/Logs/i3d_DSC_0267_log.csv"

log_df = pd.read_csv(csv_path)

print("===== Cheating Log =====")
print(log_df.head())  # show first rows

In [ ]:
from IPython.display import HTML
from base64 import b64encode

log_df = pd.read_csv(csv_path)

cap = cv2.VideoCapture(video_name)

fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

annotated_video_path = "/content/drive/MyDrive/My_Models/annotataions/i3d_0267.mp4"

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(annotated_video_path,fourcc,fps,(width,height))

tracker_to_student = {}
student_counter = 0
frame_no = 0

while True:

    ret,frame = cap.read()
    if not ret:
        break

    results = yolo.track(frame,persist=True,classes=[0],conf=0.4,verbose=False)

    if results[0].boxes.id is not None:

        boxes = results[0].boxes.xyxy.cpu().numpy()
        ids = results[0].boxes.id.cpu().numpy().astype(int)

        for box,track_id in zip(boxes,ids):

            if track_id not in tracker_to_student:
                student_counter+=1
                tracker_to_student[track_id]=student_counter

            sid = tracker_to_student[track_id]

            x1,y1,x2,y2 = map(int,box)

            current_time = frame_no/fps

            events = log_df[
                (log_df["student_id"]==sid) &
                (log_df["start_time_s"]<=current_time) &
                (log_df["end_time_s"]>=current_time)
            ]

            label="normal"
            conf=0
            color=(0,255,0)

            if len(events)>0:
                label=events.iloc[-1]["cheating_type"]
                conf=events.iloc[-1]["confidence"]

                if label!="normal":
                    color=(0,0,255)

            cv2.rectangle(frame,(x1,y1),(x2,y2),color,3)

            text=f"SID:{sid} {label} ({conf:.2f})"

            (tw,th),_ = cv2.getTextSize(text,cv2.FONT_HERSHEY_SIMPLEX,0.7,2)

            cv2.rectangle(frame,(x1,y1-35),(x1+tw+10,y1),(128,0,0),-1)

            cv2.putText(frame,text,(x1+5,y1-10),
                        cv2.FONT_HERSHEY_SIMPLEX,0.7,
                        (255,255,255),2)

    out.write(frame)
    frame_no+=1

cap.release()
out.release()

print("Annotated video saved")

In [ ]:
# Display
mp4=open(annotated_video_path,'rb').read()
data_url="data:video/mp4;base64,"+b64encode(mp4).decode()

HTML(f"""
<video width=800 controls>
<source src="{data_url}" type="video/mp4">
</video>
""")

# Person tracking issue resolve

In [ ]:
!pip install ultralytics opencv-python -q

In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO
from collections import defaultdict, deque

# =========================================
# MODELS
# =========================================
detector = YOLO("yolov8n.pt")
pose_model = YOLO("yolov8n-pose.pt")

# =========================================
# SETTINGS
# =========================================
VIDEO_PATH = "/content/drive/MyDrive/OurDataset/dataset_f5_hall-2/DSC_0129.MOV"
OUTPUT_PATH = "/content/drive/MyDrive/OurDataset/video_annotate/output_FINAL_0129.MOV"

CLIP_MEMORY = 20
SCORE_THRESHOLD = 6

# =========================================
# STORAGE
# =========================================
history = defaultdict(lambda: deque(maxlen=CLIP_MEMORY))
scores = defaultdict(float)
tracker_map = {}

student_features = {}
student_positions = {}

streaks = defaultdict(lambda: {"glance":0,"lean":0,"reach":0})

# =========================================
# RE-ID (FIXED)
# =========================================
def extract_feature(crop):
    crop = cv2.resize(crop,(64,128))
    hist = cv2.calcHist([crop],[0,1,2],None,[8,8,8],[0,256]*3)
    hist = cv2.normalize(hist,hist).flatten()
    return hist

def get_center(box):
    x1,y1,x2,y2 = box
    return np.array([(x1+x2)//2,(y1+y2)//2])

def match_student(feature, center):
    best_id = None
    best_score = 999

    for sid in student_features:

        feat_dist = np.linalg.norm(feature - student_features[sid])
        pos_dist = np.linalg.norm(center - student_positions[sid])

        # 🔥 STRICT CONDITIONS (IMPORTANT)
        if feat_dist < 0.3 and pos_dist < 150:
            if feat_dist < best_score:
                best_score = feat_dist
                best_id = sid

    return best_id

# =========================================
# RULES (UNCHANGED)
# =========================================
def side_glance(kp):
    nose, leye, reye, lear, rear = kp[0], kp[1], kp[2], kp[3], kp[4]
    if lear[2] < 0.3 or rear[2] < 0.3:
        return 1
    face_center = (leye[0] + reye[0]) / 2
    if abs(nose[0] - face_center) > 25:
        return 1
    return 0

def leaning(kp):
    ls, rs = kp[5], kp[6]
    if abs(ls[1] - rs[1]) < 35:
        return 0
    return 1

def reaching(kp):
    lw, rw = kp[9], kp[10]
    torso = (kp[5][:2] + kp[6][:2]) / 2
    if np.linalg.norm(lw[:2] - torso) > 150:
        return 1
    if np.linalg.norm(rw[:2] - torso) > 150:
        return 1
    return 0

def hand_movement(prev, kp):
    if prev is None:
        return 0
    lw, rw = kp[9], kp[10]
    plw, prw = prev[9], prev[10]
    move = np.linalg.norm(lw[:2]-plw[:2]) + np.linalg.norm(rw[:2]-prw[:2])
    if move < 30: return 0
    if move > 90: return 1
    return 0

# =========================================
# DRAW
# =========================================
SKELETON = [
    (0,1),(0,2),(1,3),(2,4),
    (5,6),(5,9),(6,10),(5,11),(6,12),(11,12)
]

def draw_pose(frame, kp, offset):
    ox, oy = offset
    for i,j in SKELETON:
        if kp[i][2]>0.3 and kp[j][2]>0.3:
            x1,y1 = int(kp[i][0])+ox, int(kp[i][1])+oy
            x2,y2 = int(kp[j][0])+ox, int(kp[j][1])+oy
            cv2.line(frame,(x1,y1),(x2,y2),(255,0,0),2)

    for i in range(len(kp)):
        if i not in [7,8]:
            x,y,c = kp[i]
            if c>0.3:
                cv2.circle(frame,(int(x)+ox,int(y)+oy),4,(0,255,255),-1)

def draw_dashboard(frame, scores):
    h,w,_ = frame.shape
    panel = 260

    overlay = frame.copy()
    cv2.rectangle(overlay,(w-panel,0),(w,h),(30,30,30),-1)
    frame[:] = cv2.addWeighted(overlay,0.6,frame,0.4,0)

    y = 40

    for sid in sorted(scores.keys()):
        score = scores[sid]
        norm = min(score/SCORE_THRESHOLD,1)
        bar = int(160*norm)

        color = (0,0,255) if score>SCORE_THRESHOLD else (0,255,0)
        status = "CHEAT" if score>SCORE_THRESHOLD else "NORMAL"

        cv2.putText(frame,f"ID {sid}",(w-panel+10,y),
                    cv2.FONT_HERSHEY_SIMPLEX,0.7,(255,255,255),2)

        cv2.rectangle(frame,(w-panel+10,y+10),
                      (w-panel+170,y+30),(80,80,80),-1)

        cv2.rectangle(frame,(w-panel+10,y+10),
                      (w-panel+10+bar,y+30),color,-1)

        cv2.putText(frame,status,(w-panel+10,y+50),
                    cv2.FONT_HERSHEY_SIMPLEX,0.6,color,2)

        y += 70

# =========================================
# MAIN
# =========================================
def run():

    cap = cv2.VideoCapture(VIDEO_PATH)

    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps == 0: fps = 25

    w = int(cap.get(3))
    h = int(cap.get(4))

    out = cv2.VideoWriter(OUTPUT_PATH,
                          cv2.VideoWriter_fourcc(*'mp4v'),
                          fps,(w,h))

    sid_counter = 0

    while True:
        ret,frame = cap.read()
        if not ret:
            break

        results = detector.track(frame,persist=True,classes=[0],conf=0.4,verbose=False)

        if results[0].boxes.id is not None:

            boxes = results[0].boxes.xyxy.cpu().numpy()
            ids = results[0].boxes.id.cpu().numpy().astype(int)

            for box,tid in zip(boxes,ids):

                x1,y1,x2,y2 = map(int,box)
                crop = frame[y1:y2,x1:x2]

                if crop.size == 0:
                    continue

                center = get_center(box)

                # ===== FIXED ID LOGIC =====
                if tid not in tracker_map:

                    feature = extract_feature(crop)
                    matched_id = match_student(feature, center)

                    if matched_id is not None:
                        sid = matched_id
                    else:
                        sid_counter += 1
                        sid = sid_counter
                        student_features[sid] = feature
                        student_positions[sid] = center

                    tracker_map[tid] = sid

                sid = tracker_map[tid]

                # update position
                student_positions[sid] = center

                # ===== POSE =====
                pose = pose_model(crop,verbose=False)
                if len(pose[0].keypoints.data)==0:
                    continue

                kp = pose[0].keypoints.data[0].cpu().numpy()
                prev = history[sid][-1] if len(history[sid])>0 else None

                # ===== RULES =====
                g = side_glance(kp)
                l = leaning(kp)
                r = reaching(kp)
                hm = hand_movement(prev,kp)

                streaks[sid]["glance"] = streaks[sid]["glance"]+1 if g else 0
                streaks[sid]["lean"] = streaks[sid]["lean"]+1 if l else 0
                streaks[sid]["reach"] = streaks[sid]["reach"]+1 if r else 0

                score = 0
                if streaks[sid]["glance"] > 20: score += 2
                if streaks[sid]["lean"] > 25: score += 1
                if streaks[sid]["reach"] > 10: score += 2
                score += hm

                scores[sid] *= 0.85
                scores[sid] += score
                scores[sid] = min(scores[sid],10)

                history[sid].append(kp)

                color = (0,0,255) if scores[sid]>SCORE_THRESHOLD else (0,255,0)

                cv2.rectangle(frame,(x1,y1),(x2,y2),color,3)
                cv2.putText(frame,f"ID:{sid} {scores[sid]:.1f}",
                            (x1,y1-10),
                            cv2.FONT_HERSHEY_SIMPLEX,0.9,color,3)

                draw_pose(frame,kp,(x1,y1))

        draw_dashboard(frame,scores)
        out.write(frame)

    cap.release()
    out.release()

    print("✅ FINAL PERFECT VERSION (NO ID DUPLICATION)")

# =========================================
run()